# Cleaning ACS Data

**Goal:**  
Prepare the ACS data so it can be used later in the housing affordability analysis.

**Plan:**
1. Load the raw ACS files
2. Combine the yearly files into one dataset
3. Check the combined data
4. Rename and organize the columns
5. Fix data types
6. Clean county and FIPS columns
7. Create useful housing variables
8. Create ACS age group data
9. Finalize ACS age group data using CBSA codes
10. Finalize ACS county-year data
11. Save the cleaned ACS datasets


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

### Step 1: Loading Raw ACS Files

In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

acs_path = project_root / "data" / "ACS"
clean_path = project_root / "clean_data"

clean_path.mkdir(exist_ok=True)

acs_files = sorted([
    file for file in acs_path.glob("acs*.csv")
    if file.name[3:7].isdigit() and "-" not in file.stem
])

if len(acs_files) == 0:
    raise FileNotFoundError("No ACS files found. Make sure files like acs2005.csv are in data/ACS.")

print("CSV files found:")
for file in acs_files:
    print(file.name)


### Step 2: Combining the Files

In [ ]:
dfs = []

for file in acs_files:
    df = pd.read_csv(file, dtype=str)
    year = int(file.name[3:7])
    df["year"] = year
    dfs.append(df)

acs = pd.concat(dfs, ignore_index=True)

acs.head()


In [ ]:
acs.shape

This shows that the ACS files were loaded correctly. Each row is one county in one year.


### Step 3: Checking the Combined Data

In [ ]:
# Check data types and non-null counts
acs.info()

In [ ]:
# Check column names
acs.columns

In [ ]:
# Check missing values
acs.isnull().sum()

### Step 4: Renaming Columns

The ACS columns use Census codes, so I renamed them to make the data easier to read and use.

In [ ]:
acs = acs.rename(columns={
    "NAME": "county_name",
    "B25007_001E": "total_occupied_units",
    "B25007_002E": "owner_occupied_total",
    "B25007_003E": "owner_occupied_hh_15_24",
    "B25007_004E": "owner_occupied_hh_25_34",
    "B25007_005E": "owner_occupied_hh_35_44",
    "B25007_006E": "owner_occupied_hh_45_54",
    "B25007_007E": "owner_occupied_hh_55_59",
    "B25007_008E": "owner_occupied_hh_60_64",
    "B25007_009E": "owner_occupied_hh_65_74",
    "B25007_010E": "owner_occupied_hh_75_84",
    "B25007_011E": "owner_occupied_hh_85_plus",
    "B25007_012E": "renter_occupied_total",
    "B25007_013E": "renter_occupied_hh_15_24",
    "B25007_014E": "renter_occupied_hh_25_34",
    "B25007_015E": "renter_occupied_hh_35_44",
    "B25007_016E": "renter_occupied_hh_45_54",
    "B25007_017E": "renter_occupied_hh_55_59",
    "B25007_018E": "renter_occupied_hh_60_64",
    "B25007_019E": "renter_occupied_hh_65_74",
    "B25007_020E": "renter_occupied_hh_75_84",
    "B25007_021E": "renter_occupied_hh_85_plus",
    "state": "state_fips",
    "county": "county_fips"
})

acs.head()


Each row is one county-year. Owner and renter columns count housing units by the age of the householder. `hh` means householder.

### Step 5: Fixing Data Types

In [ ]:
acs.dtypes

The count columns need to be numeric so they can be used for calculations later.

In [ ]:
possible_count_cols = [
    "total_occupied_units",
    "owner_occupied_total",
    "owner_occupied_hh_15_24",
    "owner_occupied_hh_25_34",
    "owner_occupied_hh_35_44",
    "owner_occupied_hh_45_54",
    "owner_occupied_hh_55_59",
    "owner_occupied_hh_60_64",
    "owner_occupied_hh_65_74",
    "owner_occupied_hh_75_84",
    "owner_occupied_hh_85_plus",
    "renter_occupied_total",
    "renter_occupied_hh_15_24",
    "renter_occupied_hh_25_34",
    "renter_occupied_hh_35_44",
    "renter_occupied_hh_45_54",
    "renter_occupied_hh_55_59",
    "renter_occupied_hh_60_64",
    "renter_occupied_hh_65_74",
    "renter_occupied_hh_75_84",
    "renter_occupied_hh_85_plus",
]

count_cols = [col for col in possible_count_cols if col in acs.columns]
missing_count_cols = [col for col in possible_count_cols if col not in acs.columns]

for col in count_cols:
    acs[col] = pd.to_numeric(acs[col], errors="coerce")

acs["year"] = pd.to_numeric(acs["year"], errors="coerce").astype(int)
acs["state_fips"] = acs["state_fips"].astype(str).str.zfill(2)
acs["county_fips"] = acs["county_fips"].astype(str).str.zfill(3)

print("Missing count columns:", missing_count_cols)

acs.dtypes


### Step 6: Cleaning County and FIPS Columns

This step cleans the county name and creates a full county FIPS code so the ACS data can merge with the other datasets later.

In [ ]:
acs["county_clean"] = (
    acs["county_name"]
    .str.replace(" County, New Jersey", "", regex=False)
    .str.strip()
)

acs["county_fips_full"] = acs["state_fips"].str.zfill(2) + acs["county_fips"].str.zfill(3)

acs[["county_name", "county_clean", "state_fips", "county_fips", "county_fips_full", "year"]].head()

### Step 7: Creating Useful ACS Variables

This step creates simple rates that are easier to compare across counties and years.

In [ ]:
acs["owner_rate"] = acs["owner_occupied_total"] / acs["total_occupied_units"]
acs["renter_rate"] = acs["renter_occupied_total"] / acs["total_occupied_units"]

acs[[
    "county_clean",
    "year",
    "total_occupied_units",
    "owner_occupied_total",
    "renter_occupied_total",
    "owner_rate",
    "renter_rate"
]].head()

### Step 8: Creating ACS Age Group Data

This step keeps the ACS age groups instead of estimating generations. Each row becomes one county, year, and age group.


In [ ]:
age_group_cols = {
    "owner_occupied_hh_15_24": ("owner", "15-24"),
    "owner_occupied_hh_25_34": ("owner", "25-34"),
    "owner_occupied_hh_35_44": ("owner", "35-44"),
    "owner_occupied_hh_45_54": ("owner", "45-54"),
    "owner_occupied_hh_55_59": ("owner", "55-59"),
    "owner_occupied_hh_60_64": ("owner", "60-64"),
    "owner_occupied_hh_65_74": ("owner", "65-74"),
    "owner_occupied_hh_75_84": ("owner", "75-84"),
    "owner_occupied_hh_85_plus": ("owner", "85+"),
    "renter_occupied_hh_15_24": ("renter", "15-24"),
    "renter_occupied_hh_25_34": ("renter", "25-34"),
    "renter_occupied_hh_35_44": ("renter", "35-44"),
    "renter_occupied_hh_45_54": ("renter", "45-54"),
    "renter_occupied_hh_55_59": ("renter", "55-59"),
    "renter_occupied_hh_60_64": ("renter", "60-64"),
    "renter_occupied_hh_65_74": ("renter", "65-74"),
    "renter_occupied_hh_75_84": ("renter", "75-84"),
    "renter_occupied_hh_85_plus": ("renter", "85+"),
}

age_group_cols = {
    col: values
    for col, values in age_group_cols.items()
    if col in acs.columns
}

age_group_rows = []

id_cols = [
    "county_clean",
    "county_fips_full",
    "year",
    "total_occupied_units",
]

for _, row in acs.iterrows():
    base_values = {col: row[col] for col in id_cols}

    for source_col, (tenure, age_group) in age_group_cols.items():
        new_row = base_values.copy()
        new_row.update({
            "age_group": age_group,
            "tenure": tenure,
            "housing_units_estimated": row[source_col],
        })
        age_group_rows.append(new_row)

acs_age_group_long = pd.DataFrame(age_group_rows)

acs_age_group_long = (
    acs_age_group_long
    .groupby(id_cols + ["age_group", "tenure"], as_index=False)["housing_units_estimated"]
    .sum()
)

acs_age_group = (
    acs_age_group_long
    .pivot_table(
        index=id_cols + ["age_group"],
        columns="tenure",
        values="housing_units_estimated",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

acs_age_group.columns.name = None

acs_age_group = acs_age_group.rename(columns={
    "owner": "owner_units_estimated",
    "renter": "renter_units_estimated",
})

for col in ["owner_units_estimated", "renter_units_estimated"]:
    if col not in acs_age_group.columns:
        acs_age_group[col] = 0

acs_age_group["total_age_group_units"] = (
    acs_age_group["owner_units_estimated"] +
    acs_age_group["renter_units_estimated"]
)

acs_age_group["age_group_share_of_total_units"] = (
    acs_age_group["total_age_group_units"] /
    acs_age_group["total_occupied_units"]
)

acs_age_group["owner_rate_within_age_group"] = np.where(
    acs_age_group["total_age_group_units"] > 0,
    acs_age_group["owner_units_estimated"] / acs_age_group["total_age_group_units"],
    np.nan
)

acs_age_group["renter_rate_within_age_group"] = np.where(
    acs_age_group["total_age_group_units"] > 0,
    acs_age_group["renter_units_estimated"] / acs_age_group["total_age_group_units"],
    np.nan
)

acs_age_group = acs_age_group[
    [
        "county_clean",
        "county_fips_full",
        "year",
        "age_group",
        "owner_units_estimated",
        "renter_units_estimated",
        "total_age_group_units",
        "age_group_share_of_total_units",
        "owner_rate_within_age_group",
        "renter_rate_within_age_group",
    ]
]

acs_age_group = acs_age_group.sort_values(
    ["year", "county_clean", "age_group"]
).reset_index(drop=True)

print("Shape:", acs_age_group.shape)

acs_age_group.head()


### Step 9: Finalizing ACS Age Group Data with CBSA Codes

This groups New Jersey counties into CBSAs. The CBSA totals only include New Jersey counties, not the out-of-state counties.


In [ ]:
county_to_cbsa = {
    "Atlantic": ("12100", "Atlantic City-Hammonton, NJ"),
    "Bergen": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Burlington": ("37980", "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"),
    "Camden": ("37980", "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"),
    "Cape May": ("36140", "Ocean City, NJ"),
    "Cumberland": ("47220", "Vineland-Bridgeton, NJ"),
    "Essex": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Gloucester": ("37980", "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"),
    "Hudson": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Hunterdon": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Mercer": ("45940", "Trenton-Princeton, NJ"),
    "Middlesex": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Monmouth": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Morris": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Ocean": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Passaic": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Salem": ("37980", "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD"),
    "Somerset": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Sussex": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Union": ("35620", "New York-Newark-Jersey City, NY-NJ-PA"),
    "Warren": ("10900", "Allentown-Bethlehem-Easton, PA-NJ"),
}

acs_age_group["cbsa_code"] = acs_age_group["county_clean"].map(lambda x: county_to_cbsa[x][0])
acs_age_group["cbsa_name"] = acs_age_group["county_clean"].map(lambda x: county_to_cbsa[x][1])

acs_age_group_cbsa = (
    acs_age_group
    .groupby(["cbsa_code", "cbsa_name", "year", "age_group"], as_index=False)
    .agg({
        "owner_units_estimated": "sum",
        "renter_units_estimated": "sum",
        "total_age_group_units": "sum",
    })
)

cbsa_year_totals = (
    acs_age_group_cbsa
    .groupby(["cbsa_code", "year"])["total_age_group_units"]
    .transform("sum")
)

acs_age_group_cbsa["age_group_share_of_total_units"] = (
    acs_age_group_cbsa["total_age_group_units"] / cbsa_year_totals
)

acs_age_group_cbsa["owner_rate_within_age_group"] = np.where(
    acs_age_group_cbsa["total_age_group_units"] > 0,
    acs_age_group_cbsa["owner_units_estimated"] / acs_age_group_cbsa["total_age_group_units"],
    np.nan
)

acs_age_group_cbsa["renter_rate_within_age_group"] = np.where(
    acs_age_group_cbsa["total_age_group_units"] > 0,
    acs_age_group_cbsa["renter_units_estimated"] / acs_age_group_cbsa["total_age_group_units"],
    np.nan
)

acs_age_group_cbsa = acs_age_group_cbsa[
    [
        "cbsa_code",
        "cbsa_name",
        "year",
        "age_group",
        "owner_units_estimated",
        "renter_units_estimated",
        "total_age_group_units",
        "age_group_share_of_total_units",
        "owner_rate_within_age_group",
        "renter_rate_within_age_group",
    ]
]

acs_age_group_cbsa = acs_age_group_cbsa.sort_values(
    ["year", "cbsa_name", "age_group"]
).reset_index(drop=True)

print("Shape:", acs_age_group_cbsa.shape)

acs_age_group_cbsa.head()


### Step 10: Finalizing ACS County-Year Data

This creates the county-year ACS file for the main merge. It keeps the 21 New Jersey counties.


In [ ]:
acs_county_year = acs[acs["state_fips"].astype(str).str.zfill(2) == "34"].copy()

acs_county_year = acs_county_year[
    [
        "county_fips_full",
        "county_clean",
        "year",
        "total_occupied_units",
        "owner_occupied_total",
        "renter_occupied_total",
        "owner_rate",
        "renter_rate",
    ]
]

acs_county_year = acs_county_year.rename(columns={
    "county_fips_full": "county_fips",
    "county_clean": "county"
})

acs_county_year["county_fips"] = acs_county_year["county_fips"].astype(str).str.zfill(5)

acs_county_year = acs_county_year.sort_values(
    ["year", "county"]
).reset_index(drop=True)

print("Shape:", acs_county_year.shape)
print("Counties:", acs_county_year["county"].nunique())
print("Duplicates:", acs_county_year.duplicated(subset=["county_fips", "year"]).sum())

acs_county_year.head()


### Step 11: Saving Data

This step saves the two cleaned ACS datasets.


In [ ]:
acs_age_group_cbsa.to_csv(
    clean_path / "ACS_generation.csv",
    index=False
)

acs_county_year.to_csv(
    clean_path / "ACS_county.csv",
    index=False
)

